# 23rd Aug 2026

In [4]:
import pandas as pd
from nltk.tokenize import word_tokenize
import unicodedata
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
import torch
from torch import nn,optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
s_w = stopwords.words('english')
def clean_text(text):
    text = str(text).lower()
    text = text.strip()
    text = unicodedata.normalize("NFKC",text)
    #http://,https://,www,
    text = re.sub(r"http://\S+|https://\S+|www\S+","",text) # remove urls
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9._%+-]+.[A-Za-z0-9._%+-]+","",text)# removing emails
    text = re.sub(r"@\w+","",text)
    text = re.sub(r"#\w+","",text)
    text = re.sub(r"[^A-Za-z\s]+","",text)
    tokens = word_tokenize(text)
    cleaned_tokens = []
    #stemmer = PorterStemmer()
    lemmatizer = WordNetLemmatizer()
    for token in tokens:
        if (token not in s_w) and (len(token)>2):
            #token = stemmer.stem(token)
            token = lemmatizer.lemmatize(token)
            cleaned_tokens.append(token)

    cleaned_text = " ".join(cleaned_tokens)
    return cleaned_text
df = pd.read_csv(r"C:\Users\barat\OneDrive\Documents\GitHub\Data-Science\Data\Suicide_Ideation_Dataset(Twitter-based).csv")
df['Tweet'] = df['Tweet'].apply(clean_text)
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word_to_index = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for sentence in df['Tweet']:
    for token in sentence.split():
        if token not in word_to_index:
            word_to_index[token] = len(word_to_index)
X = []
max_len = max([len(doc.split()) for doc in df["Tweet"]])
for sentence in df["Tweet"]:
    encode = []
    for token in sentence.split():
        if token in word_to_index.keys():
            encode.append(word_to_index[token])
        else:
            encode.append(word_to_index[UNK_TOKEN])
    encode =encode+ [word_to_index[PAD_TOKEN]] * (max_len - len(encode))
    X.append(encode)
X = torch.tensor(X, dtype=torch.long)
encoder = LabelEncoder()
y = torch.tensor(encoder.fit_transform(df["Suicide"]),dtype=torch.float32).view(-1,1)
x_train,x_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class TextClassification(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassification, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        output,(hidden,cell) = self.lstm(embedded)
        hidden = hidden[-1]
        output = self.sigmoid(self.fc(hidden))
        return output
model = TextClassification(len(word_to_index), 100, 128, 1)
optimizer = optim.Adam(model.parameters())
criterion = nn.BCELoss()

num_epoch = 1000
for epoch in range(1,num_epoch+1):
    pred = model(x_train)
    loss = criterion(pred,y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch%100==0:
        print(f"{epoch}/{num_epoch}--> Loss--> {loss.item()}")
pred = model(x_train)
pred = (pred>0.5).float().detach().numpy()
y_train = y_train.detach().numpy()

print(f"Accuracy: {accuracy_score(y_train,pred)}")
print(f"Precision: {precision_score(y_train,pred)}")
print(f"Recall: {recall_score(y_train,pred)}")
print(f"F1 Score: {f1_score(y_train,pred)}")

pred = model(x_test)
pred = torch.argmax(pred,dim=1)
pred = (pred>0.5).float().detach().numpy()
y_test = y_test.detach().numpy()
print(f"Accuracy: {accuracy_score(y_test,pred)}")
print(f"Precision: {precision_score(y_test,pred)}")
print(f"Recall: {recall_score(y_test,pred)}")
print(f"F1 Score: {f1_score(y_test,pred)}")

100/1000--> Loss--> 0.6547886729240417
200/1000--> Loss--> 0.6546230912208557
300/1000--> Loss--> 0.6545649170875549
400/1000--> Loss--> 0.6545461416244507
500/1000--> Loss--> 0.6545385122299194
600/1000--> Loss--> 0.6545343399047852
700/1000--> Loss--> 0.6545318961143494
800/1000--> Loss--> 0.6545302271842957
900/1000--> Loss--> 0.6545290350914001
1000/1000--> Loss--> 0.6545281410217285
Accuracy: 0.6375087473757872
Precision: 1.0
Recall: 0.0019267822736030828
F1 Score: 0.0038461538461538464
Accuracy: 0.6061452513966481
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


c:\Users\barat\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
